In [1]:
from dataclasses import dataclass
from typing import TypedDict, Annotated, Literal

from dotenv import  load_dotenv
from langchain_core import messages
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langchain_experimental.graph_transformers.llm import system_prompt
from langgraph.graph import StateGraph,START,END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.checkpoint.postgres import  PostgresSaver
from langgraph.graph.message import MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 実行時のコンテキストを定義
@dataclass
class UserContext:
    username:str
    membership_level:str

#2. 状態を定義
class OverAllState(MessagesState):
    user_input:str
    output:str

#3. ノードを定義
def llm_node(state:OverAllState,runtime:Runtime[UserContext]) -> OverAllState:
    #1. コンテキストを取得し、現在のユーザーの会員ランクを判定
    runtime_context = runtime.context

    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username
        logger.info(f"現在のユーザー:{username}、会員ランク:{level}")

        if level == "VIP":
            system_prompt = f"あなたは上級カスタマーアシスタントです。現在のVIPユーザーは{username}さんです。敬語を使い、丁寧で心のこもった口調で対応し、返信の最後に'VIP🏅サービス'と付け加えてください"
        else:
            system_prompt = f"あなたは一般カスタマーアシスタントです。現在のユーザーは{username}さんです。フレンドリーかつ簡潔に質問へ回答してください"

    else:
        system_prompt = f"あなたは一般カスタマーアシスタントです。フレンドリーかつ簡潔に質問へ回答してください"

    user_input = state["user_input"]
    messages = state.get("messages",[])
    response = model.invoke([SystemMessage(content=system_prompt)]+ messages + [HumanMessage(content=user_input)]).content
    return {
        "messages" : messages,
        "output":response
    }

#4. グラフを構築
builder = StateGraph(state_schema=OverAllState,context_schema=UserContext)

builder.add_node("llm_node",llm_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()

# ==========1回目の呼び出し: VIPユーザーのコンテキストを渡す====================
res = graph.invoke(
    {"user_input":"こんにちは、最近どんなキャンペーンがあるか調べてください"},
    context=UserContext(username="Alice",membership_level="VIP")
)
print(res)




2026-08-15 15:58:33.556 | INFO     | __main__:llm_node:45 - 現在のユーザー:Alice、会員ランク:VIP


{'messages': [], 'user_input': 'こんにちは、最近どんなキャンペーンがあるか調べてください', 'output': 'こんにちは、Aliceさん。ご連絡いただきありがとうございます😊  \n現在開催中のキャンペーンについて、すぐに詳しい情報を確認いたしますので、少々お待ちくださいませ。  \n具体的な内容が分かり次第、こちらでご案内いたしますね。  \n何かご希望やご質問がございましたら、遠慮なくお申し付けください。  \nVIP🏅サービス'}


In [3]:
# ==========2回目の呼び出し: 一般ユーザーのコンテキストを渡す====================
res1 = graph.invoke(
    {"user_input":"こんにちは、最近どんなキャンペーンがあるか調べてください"},
    context=UserContext(username="Alice",membership_level="一般ユーザー")
)
print(res1)

2026-08-15 15:59:43.481 | INFO     | __main__:llm_node:45 - 現在のユーザー:Alice、会員ランク:一般ユーザー


{'messages': [], 'user_input': 'こんにちは、最近どんなキャンペーンがあるか調べてください', 'output': 'こんにちは、Aliceさん！  \n具体的なキャンペーン情報をお調べするには、どのサービスやジャンル（通販、旅行、飲食、エンタメなど）をご希望ですか？  \nまた、可能であれば「期間」や「予算」なども教えていただけると、より的確にご案内できますよ。'}
